# Corovans — will the player come back after the first game?

Side question to the main project (`notebook.ipynb`). One row = the first game of a player,
target `returned` = the player started at least one more game. Built by `prepare_retention.py`.

Limitations of the data:

- a session is uploaded only when the player dies, so players who closed the tab in the middle
  of the first game are invisible;
- sessions without `playerId` (early May builds) are dropped — they can't be linked to later games;
- first games younger than 1 day are dropped — returns usually happen within minutes.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data/first_games.csv')
df.shape

(383, 35)

In [2]:
df.returned.mean()

0.2924281984334204

## When did players come

In [3]:
df.groupby(df.started_at.str[:7]).returned.agg(['count', 'mean']).round(2)

,count,mean
started_at,,
2026-05,350,0.29
2026-06,7,0.14
2026-07,7,0.00
2026-08,3,0.33
2026-09,1,0.00
2026-10,15,0.47


Most first games (350 of 383) are from May, the launch month. Later months add only a few players each.

## Return rate by how the first game went

In [4]:
waves = pd.cut(df.wave_reached, [0, 4, 7, 10, 100], labels=['1-4', '5-7', '8-10', '11+'])
df.groupby(waves, observed=True).returned.agg(['count', 'mean']).round(2)

,count,mean
wave_reached,,
1-4,114,0.27
5-7,139,0.29
8-10,80,0.32
11+,50,0.30


In [5]:
df.groupby('has_name').returned.agg(['count', 'mean']).round(2)

,count,mean
has_name,,
0,185,0.26
1,198,0.32


In [6]:
df.groupby('death_source').returned.agg(['count', 'mean']).round(2)

,count,mean
death_source,,
arrow,42,0.36
fire-arrow,4,0.25
fire-burn,3,0.00
guard,334,0.29


## Single-feature AUC

In [7]:
numerical = [c for c in df.select_dtypes('number').columns if c != 'returned']
auc = pd.Series({c: roc_auc_score(df.returned, df[c]) for c in numerical})
auc.sort_values(ascending=False).round(3)

dashes              0.546
has_name            0.538
dashes_per_min      0.537
flawless_waves      0.522
last_wave_s         0.522
no_damage_waves     0.518
gold_spent          0.518
last_wave_damage    0.513
caravans_robbed     0.512
wave_reached        0.511
waves_completed     0.511
shop_visits         0.510
avg_wave_s          0.509
gold_earned         0.508
cards_picked        0.508
guards_killed       0.507
cards_distinct      0.505
final_score         0.503
rebirths            0.503
rerolls             0.502
relics              0.500
hits_taken          0.500
pickups             0.499
damage_taken        0.498
duration_s          0.497
caravans_escaped    0.493
boss_death          0.491
got_mercy_heal      0.491
cards_rare_plus     0.477
hour_utc            0.473
dtype: float64

## Models vs shuffled labels

5-fold CV repeated 5 times. To see what "no signal" looks like on 383 rows,
the same models are also scored on randomly shuffled labels.

In [8]:
X = df[numerical]
y = df.returned
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=1)
models = {
    'logistic regression': make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000)),
    'random forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=1),
}
y_shuffled = y.sample(frac=1, random_state=1).values

rows = []
for name, model in models.items():
    real = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    fake = cross_val_score(model, X, y_shuffled, cv=cv, scoring='roc_auc')
    rows.append({'model': name, 'auc': real.mean(), 'std': real.std(), 'auc_shuffled': fake.mean()})
pd.DataFrame(rows).round(3)

,model,auc,std,auc_shuffled
0,logistic regression,0.487,0.076,0.434
1,random forest,0.459,0.065,0.476


## Conclusions

- How the first game went (waves reached, damage, cards, gold) does not tell whether the player
  comes back: single features give AUC 0.47–0.55, models are on the level of shuffled labels.
- The decision to come back is probably driven by things the telemetry doesn't log: where the
  player came from, device, whether they just wanted a look at the launch.
- I'm adding a short receipt sent when a game starts, so abandoned first games become visible.
  This question is worth revisiting when more players arrive.
- The main project therefore stays on predicting death in a wave, where the signal is strong.